# 1D boundaries

`Fluid1D` resolves a plasma along one dimension: drift-diffusion of every species, the
electron energy and Poisson's equation, all implicit. This tutorial looks at its
boundaries and at ions with inertia: a presheath that accelerates the ions to the
sound speed (the Bohm criterion), the wall's electron flux of a resolved sheath, and an
outflow such as the fluid side of a fluid-PIC coupling. The transport coefficients are
synthetic; every case runs in seconds.

## 1. Ions with momentum

A uniform source of argon ions and electrons between a symmetry plane (left) and an
absorbing wall (right), isothermal electrons at 2 eV. With `momentum=True` the ions obey
their momentum equation instead of drift-diffusion, and a large mobility makes them
nearly collisionless. The source is chosen so that the sonic point sits near the wall.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr
from plasmacoll.constants import ELEMENTARY_CHARGE as e

from wellbalanced import (
    ELECTRON,
    Boundary,
    Fluid1D,
    FluidSpecies,
    PotentialBoundary,
    Species,
)

argon_ion = Species.atomic("Ar+", 39.948, 1)
te, tg, length, density = 2.0, 0.025, 0.05, 1.0e15
sound = np.sqrt(e * (te + tg) / argon_ion.mass)
source = density * sound / (2.0 * length)  # m^-3 s^-1


def presheath(thermal_share):
    """The slab with a wall that takes ``thermal_share`` of the electrons' thermal flux."""
    model = Fluid1D(
        species=[
            FluidSpecies(ELECTRON, mobility=1.0e4),
            FluidSpecies(argon_ion, mobility=1.0e3, momentum=True),
        ],
        reactions=[],
        length=length,
        cells=400,
        electron_temperature=te,
        gas_temperature=tg,
        sources={"e": source, "Ar+": source},
        boundaries={
            "left": {"e": Boundary.symmetry(), "Ar+": Boundary.symmetry()},
            "right": {"e": Boundary.wall(thermal_share=thermal_share)},
        },
        potential={"left": PotentialBoundary.field(0.0)},
    )
    guess = density * (1.0 - 0.5 * (model.x / length) ** 2)
    phi = te * np.log(guess / guess[-1]) + 5.0 * te
    start = model.pack(
        {"e": guess, "Ar+": guess}, te, phi, {"Ar+": source * model.faces}
    )
    return model, model.steady_state(start)


model, state = presheath(0.5)
state.converged

Every ion made leaves through the wall, and as many electrons: the wall floats.

In [ ]:
made = source * length
state.fluxes["Ar+"][-1] / made, state.fluxes["e"][-1] / made

The ion velocity lives on the faces between cells (`state.velocity`). The ions reach the
sound speed a few Debye lengths before the wall and then go supersonic into the sheath,
where the densities separate.

In [ ]:
mach = xr.DataArray(
    state.velocity("Ar+") / sound,
    coords={"x": ("x", model.faces * 1e3, {"long_name": "x", "units": "mm"})},
    name="mach",
    attrs={"long_name": "ion Mach number", "units": "1"},
)
profiles = state.to_dataset()
profiles = profiles.assign_coords(x=profiles.x * 1e3)
profiles.x.attrs.update(long_name="x", units="mm")
with plasma_plots.figure(1, 3, figsize=(12, 3.6)) as fig:
    profiles.density.plasma.plot.profiles(x="x", over="species", ax=fig[0])
    mach.plasma.plot.lineout(ax=fig[1])
    profiles.potential.plasma.plot.lineout(ax=fig[2])

## 2. The kinetic wall

`Boundary.wall(thermal_share=s)` takes `s v_th n` of the electrons in the cell next to the
wall. `s = 1/2` (Hagelaar et al. 2000, the default) counts that density as a
half-Maxwellian leaving; `s = 1/4` as a full Maxwellian, as kinetic sheath theory and the
global model do when the fluid resolves the sheath. With Boltzmann electrons the floating
sheaths of the two differ by $T_e \ln 2$.

In [ ]:
_, kinetic = presheath(0.25)
sheaths = {"s = 1/2": state, "s = 1/4": kinetic}
{name: s.potential[0] for name, s in sheaths.items()}  # V over the wall

In [ ]:
(state.potential[0] - kinetic.potential[0]) / (te * np.log(2.0))

## 3. An outflow

`Boundary.outflow(speed)` lets every species leave at `speed` times the density next to the
end: the fluid side of a coupling to a particle code that hands over its flux over its
density. A prescribed flux would leave the level of the plasma free; a speed fixes it, so
halving the speed doubles the density at the end.

In [ ]:
def outflow(speed):
    model = Fluid1D(
        species=[FluidSpecies(ELECTRON, 30.0), FluidSpecies(argon_ion, 10.0)],
        reactions=[],
        length=0.02,
        cells=100,
        electron_temperature=te,
        sources={"e": 1.0e20, "Ar+": 1.0e20},
        boundaries={
            "left": {"e": Boundary.symmetry(), "Ar+": Boundary.symmetry()},
            "right": {"e": Boundary.outflow(speed), "Ar+": Boundary.outflow(speed)},
        },
        potential={"left": PotentialBoundary.field(0.0)},
    )
    guess = np.full(model.cells, 1.0e20 * 0.02 / speed)
    return model.steady_state(model.pack({"e": guess, "Ar+": guess}, te, 0.0))


speeds = [500.0, 1000.0]
outflows = xr.concat(
    [outflow(v).to_dataset() for v in speeds], dim="speed"
).assign_coords(speed=("speed", speeds, {"long_name": "outflow speed", "units": "m/s"}))
outflows.density.sel(species="Ar+").isel(x=-1).values

In [ ]:
with plasma_plots.figure(1, 1, figsize=(5, 3.6)) as fig:
    outflows.density.sel(species="Ar+").plasma.plot.profiles(
        x="x", over="speed", ax=fig[0]
    )

Electrons take `energy_factor` times $T_e$ each through an outflow (`Boundary.outflow(speed,
energy_factor)`, 2 for a Maxwellian flux); with the energy equation solved, a larger factor
makes a colder plasma. Boundaries can also be changed between time steps with
`model.set_boundary(end, species, boundary)`, as a coupling does.